# Portfolio vs Nifty 50 — prototype
Runs the pure core (`benchmark_service.build_comparison_series`) on the real TRI history + synthetic transactions, then plots the two cumulative money-weighted return lines.

Customer XIRR is intentionally not computed here — the backend reuses `xirr_service.compute_portfolio_xirr`. This service/notebook owns the chart + benchmark XIRR.

In [ ]:
import sys, csv
from datetime import date
sys.path.insert(0, '..')
from app.domains.portfolio.services import benchmark_service as bs

# Real Nifty 50 TRI from the validated Part-1 CSV; nearest-on-or-before lookup.
rows = []
with open('../nifty50_tri_full.csv') as fh:
    for r in csv.DictReader(fh):
        rows.append((date.fromisoformat(r['tri_date']), float(r['tri_value'])))
tri_lookup = bs.build_step_lookup(rows)
print('TRI rows:', len(rows), '| 2024-01-31:', tri_lookup(date(2024, 1, 31)))

In [ ]:
# Synthetic monthly-ish SIP into one scheme; illustrative NAV grown 12%/yr.
txns = [
    bs.TxnLite(date(2023, 1, 1), 'BUY', 10000.0, 1000.0, 'S1'),
    bs.TxnLite(date(2023, 7, 1), 'BUY', 10000.0, 950.0, 'S1'),
    bs.TxnLite(date(2024, 1, 1), 'BUY', 10000.0, 900.0, 'S1'),
]

def nav_lookup(scheme, on):
    base = date(2023, 1, 1)
    yrs = (on - base).days / 365.0
    return 10.0 * (1.12 ** yrs)

res = bs.build_comparison_series(txns, nav_lookup, tri_lookup, as_of=date(2024, 6, 1), horizon='MAX')
print('summary:', res.summary)
print('points:', len(res.dates),
      '| last customer %:', round(res.customer_pct[-1] * 100, 2),
      '| last nifty %:', round(res.benchmark_pct[-1] * 100, 2))

In [ ]:
# Plot (requires matplotlib; guarded so headless execution never hard-fails).
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(res.dates, [p * 100 for p in res.customer_pct], label='Customer')
    plt.plot(res.dates, [p * 100 for p in res.benchmark_pct], '--', label='Nifty 50')
    plt.axhline(0, color='gray', lw=0.5)
    plt.ylabel('cumulative return %')
    plt.legend()
    plt.title('Portfolio vs Nifty 50')
    plt.show()
except ImportError:
    print('matplotlib not installed; run `pip install matplotlib` to see the chart.')